# Linear Regression (ordinary least squares)
                          
                             (i)  Ridge Regression

                             (ii) Lasso Regression  (Coordinate Descent, Soft Thresholding Operator)

In [62]:
import numpy as np
import matplotlib.pyplot as plt

class RidgeRegression():
    def __init__(self,lambd=1.0, method='solve'):
        self.method = method
        self.lambd = lambd


    def fit(self, X, Y):
        N, D = X.shape  # number of samples and features
        if self.method == 'solve':
            matrix1 = X.T@X + self.lambd*np.eye(D)
            return np.linalg.solve(matrix1, X.T@Y)
        elif self.method == 'svd':
            U, S, VT = np.linalg.svd(X, full_matrices=False)
            d_factor = S/(S**2 + self.lambd)
            UT_Y = U.T @ Y
            if UT_Y.ndim > 1:
                d_factor = d_factor[:, None]
                
            return VT.T @ (d_factor * UT_Y)

    def predict(self, X, Beta):
        return X@Beta

class LassoRegression():
    def __init__(self, lambd=1.0, tol=1e-5, max_iter=10000):
        self.lambd = lambd
        self.tol = tol
        self.max_iter = max_iter

    def fit(self, X, Y):

        N, D = X.shape  # number of samples and features

        beta = np.zeros(D)

        residual = Y - X@beta

        sample_nrmlized_featr_val  = (1/N) * np.sum(X**2, axis=0)

        for iteration in range(self.max_iter):
            old_beta = beta.copy()
            for k in range(D):
                if sample_nrmlized_featr_val[k] == 0:
                    continue

                X_k = X[:,k]
                rho_k = (1/N)*np.dot(X_k, residual)+ beta[k]*sample_nrmlized_featr_val[k] 

                if rho_k>= self.lambd:
                    beta_new = (rho_k-self.lambd)/sample_nrmlized_featr_val[k]
                elif rho_k<=-self.lambd:
                    beta_new = (rho_k + self.lambd)/sample_nrmlized_featr_val[k]

                else:
                    beta_new = 0

                if beta_new != beta[k]:
                    residual += X_k*(beta[k]-beta_new)
                    beta[k] = beta_new
     

            if np.max(np.abs(old_beta-beta)) < self.tol:
                break


        return beta

    
    def predict(self, X, beta):
        return X@beta


def data_generator(N=1000, D=10, reduced_dim = 5,noise_std=0.1):
    np.random.seed(0)
    X = np.random.randn(N, D)
    true_beta = np.zeros(D)
    true_beta[:reduced_dim] = np.random.randn(reduced_dim)
    Y = X @ true_beta + noise_std * np.random.randn(N)
    return X, Y, true_beta


X, Y, true_beta = data_generator()


lasso = LassoRegression(lambd=0.02)
beta = lasso.fit(X, Y)
predicted_y = lasso.predict(X, beta)

print("Lasso Mean Squared Error:", np.mean((predicted_y - Y)**2))



beta = RidgeRegression(method='solve').fit(X,Y)
predicted_y = RidgeRegression().predict(X,beta)

print("Ridge (solve)Mean Squared Error:", np.mean((predicted_y - Y)**2))





beta = RidgeRegression(method='svd').fit(X,Y)
predicted_y = RidgeRegression().predict(X,beta)

Error = np.mean((predicted_y - Y)**2)
print("Ridge (svd) Mean Squared Error:", Error)


Lasso Mean Squared Error: 0.012053391435646762
Ridge (solve)Mean Squared Error: 0.009959680344770074
Ridge (svd) Mean Squared Error: 0.00995968034477007


# Logistic Regression

In [59]:
from sklearn.datasets import make_classification
import numpy as np


class logistic_regression():
    def __init__(self, solver = 'newton', reg = 1e-4, max_iter = 1000, tol = 1e-5, lr = 0.1):

        self.solver = solver
        self.regularizer = reg
        self.max_iter = max_iter
        self.tol = tol
        self.lr = lr
        self.weights = None


    def _sigmoid_function(self, z):

        return np.where(z>0, 1/(1 + np.exp(-z)), np.exp(z)/(1 + np.exp(z)))


    def _loss_function(self, Y, p):

        p = np.clip(p,1e-12, 1-1e-12)

        return -np.mean(Y*np.log(p) + (1-Y)*np.log(1-p)) + self.regularizer*np.sum((self.weights[1:]**2))


    def fit(self, X, Y):

        N, D = X.shape

        aux_vector = np.full((N,1), 1.0)
        X_full = np.concatenate([aux_vector, X], axis = 1)
        Y = Y.reshape(-1, 1)
        
       
        self.weights = np.zeros((D+1, 1))
        
        if self.solver=='newton':
            self._fit_newton(X_full, Y, N, D)

        elif self.solver == 'gd':
            self._fit_gd(X_full, Y, N, D)

        else:
            print("solver should be newton or gd")

        return self


    def _fit_newton(self, X, Y, N, D):
        regularization_matrix = self.regularizer*np.eye(D+1)
        regularization_matrix[0,0] = 0.0

        for iteration in range(self.max_iter):
            Z = X@self.weights
            p = self._sigmoid_function(Z)
            loss = self._loss_function(Y, p)

            Grad = (-1/N)*(X.T@(Y-p)) + 2*regularization_matrix@self.weights
            pp = np.clip((p*(1-p)), 1e-8, 0.25).squeeze()
            Hess = (1/N)*((X.T*pp)@(X)) + 2*regularization_matrix
            delta = np.linalg.solve(Hess, Grad)
            
            self.weights-=delta

            if np.linalg.norm(delta) < self.tol:
                break

        return self.weights



    def _fit_gd(self, X, Y, N, D):
        regularization_matrix = self.regularizer*np.eye(D+1)
        regularization_matrix[0,0] = 0.0
       
        for iteration in range(self.max_iter):
            Z = X@self.weights
            p = self._sigmoid_function(Z)
            loss = self._loss_function(Y, p)
            Grad = (-1/N)*(X.T@(Y-p)) + 2*regularization_matrix@self.weights

            self.weights-=self.lr*Grad

            if np.linalg.norm(Grad) < self.tol:
                break

        return self.weights


    def predict(self, X_data, threshold = 0.5):
        N, D = X_data.shape
        X_full = np.concatenate([np.ones((N, 1)), X_data], axis=1)
        return (self._sigmoid_function(X_full@self.weights) >= threshold).astype(int)
    
        
X_data, y_data = make_classification(
        n_samples=2000,
        n_features=10,
        n_informative=8,
        n_redundant=2,
        random_state=41,
    )



model = logistic_regression(solver = 'newton')
model.fit(X_data, y_data)

predictions = model.predict(X_data)
print(f"accuracy: {np.mean(predictions.ravel() == y_data)}")

accuracy: 0.8845
